# Molab 03: Timing Attacks, Logical-Clock Defense (NB5) & Final Merge

**Goal:** Test RQ4 — can an adversary who only controls *when* transactions are submitted suppress detection, and does evidence built on logical-clock order (what re-timing cannot change) resist this where wall-clock evidence does not? Then run the project's final merge step: unseal every dataset whose confirmatory sweep is complete, compute every frozen hypothesis's verdict, and produce the benchmark comparison against the published baseline.

**Covers:** NB5 in full (logical/combined detectors, burst/stretch/blend attacks with matched displacement controls, 6 feature ablations, GFP baseline where `snapml` installs, CPU service time), **and** the merge step (`amlh.merge.merge`, the same function `notebooks/merge.ipynb` runs) — this is the one place sealed test metrics are ever computed, and every unsealing is permanently logged.

**This notebook is useful standalone, and more complete when chained after 01 and 02.** By default its merge step reads this session's own NB5 output plus whatever is already committed in `main`'s `results/` (today: PC-8's finished NB1 seed and all of PC-18's finished NB3 TD-GNN work), so even a lone run produces a real benchmark table. Set `EXTRA_ROOTS` below to also fold in notebooks 01's and 02's fresh output (copy their unzipped working folders next to this one, or push them to a branch).

**Expected wall time on the stated hardware:** 1 to 4 hours for NB5's P0 (roughly 150 LightGBM fits/attacks on HI-Small, similar per-job cost to notebook 02), then minutes for the merge step regardless of how much data feeds it. **No GPU needed or used.**

## What makes this notebook self-contained
Every cell below is idempotent and safe to re-run. On a fresh Molab session it, in order: (1) clones `aml-stream`'s
`main` branch from GitHub (public, no token needed — set `GITHUB_TOKEN` as an environment variable/secret only if the
repository has since been made private); (2) downloads the AMLworld CSVs it needs directly from Kaggle's public
download endpoint (no `kaggle.json` needed, with a hash check against `docs/SPEC.md`'s verified values, because one
download endpoint has silently truncated a file before); (3) installs the exact pinned dependencies; (4) runs the
test suite as a gate — **a failing test stops the notebook before it spends any compute**; (5) reuses whatever
earlier, *verified* output already exists in the cloned repository (so PC-8's and PC-18's already-finished jobs are
never repeated); (6) runs its job list under a wall-clock budget that always stops cleanly before the 12 h session
limit, in a way that is safe to interrupt and resume (`Run all` again after a restart continues from the last
checkpoint, it does not start over); (7) writes a human-readable analysis with tables and plots; (8) zips the output
folder so it survives the session ending.

**Nothing here needs you to upload or attach anything by hand.**

In [ ]:
# ============================= Settings: the only cell you would normally change =============================
import os, sys, time
NOTEBOOK = 'NB5'
REPO_URL = os.environ.get('AML_REPO_URL', 'https://github.com/Shahoud867/aml-stream.git')
REPO_BRANCH = os.environ.get('AML_REPO_BRANCH', 'main')   # pin a commit instead by setting this to a SHA
BUDGET_HOURS = float(os.environ.get('AML_BUDGET_HOURS', 11.0))   # stops cleanly well inside Molab's 12 h cap
TIERS = ('P0', 'P1')
RUN_TESTS = True     # a failing test stops the notebook before it spends any compute
INSTALL = True

SESSION_START = time.time()
def hours_left():
    return BUDGET_HOURS - (time.time() - SESSION_START) / 3600


In [ ]:
# ============================= Where things live =============================
# A plain, dependency-free layout that works on any Jupyter kernel (Molab, a local machine, Colab, ...): everything
# lives under the user's home directory, so no platform-specific mount point is assumed.
from pathlib import Path

BASE = Path(os.environ.get('AML_WORKDIR', Path.home() / 'aml_molab'))
BASE.mkdir(parents=True, exist_ok=True)
CODE = BASE / 'aml-stream'               # the cloned repository
INPUT = BASE / 'data'                    # raw AMLworld files
CACHE = BASE / 'cache'                   # prepared (parquet) data, rebuilt once and reused across cells
WORKING = BASE / f'out_{NOTEBOOK.lower()}'
WORKING.mkdir(parents=True, exist_ok=True)
for p in (INPUT, CACHE):
    p.mkdir(parents=True, exist_ok=True)
print('workdir', BASE)
print('  code   ->', CODE)
print('  data   ->', INPUT)
print('  cache  ->', CACHE)
print('  output ->', WORKING)

In [ ]:
# ============================= Get the code =============================
# Clones (or fast-forwards) the repository. The expected code hash is computed from whatever we actually cloned,
# never hard-coded, so this notebook can never go stale the way a fixed-hash code package would.
import subprocess

def run(cmd, **kw):
    r = subprocess.run(cmd, text=True, capture_output=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stdout}\n{r.stderr}")
    return r.stdout

token = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
clone_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
if not (CODE / '.git').exists():
    print('cloning', REPO_URL, '...')
    run(['git', 'clone', '--branch', REPO_BRANCH, clone_url, str(CODE)])
else:
    print('updating existing clone ...')
    run(['git', '-C', str(CODE), 'fetch', 'origin', REPO_BRANCH])
    run(['git', '-C', str(CODE), 'checkout', REPO_BRANCH])
    run(['git', '-C', str(CODE), 'reset', '--hard', f'origin/{REPO_BRANCH}'])

sys.path.insert(0, str(CODE / 'src'))
from amlh.runner import code_hash
CODE_HASH = code_hash(CODE)
print('HEAD', run(['git', '-C', str(CODE), 'rev-parse', 'HEAD']).strip())
print('code hash', CODE_HASH)

In [ ]:
# ============================= Install dependencies =============================
if INSTALL:
    run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-kaggle.txt')])
    print('requirements-kaggle.txt installed')


In [ ]:
# ============================= Raw data =============================
# Downloaded directly from Kaggle's public endpoint (no account needed) and checked against the row counts and
# SHA-256 hashes in docs/SPEC.md -- the same check every notebook in this project runs, because one Kaggle download
# endpoint has silently truncated a file before while still reporting success.
from amlh.fetch import ensure_raw
from amlh.session import verify_raw

DATASETS = ['HI-Small', 'LI-Small']
print('fetching', DATASETS)
ensure_raw(DATASETS, INPUT)
for d in DATASETS:
    print(d, verify_raw(d, INPUT))

In [ ]:
# ============================= Tests: the gate =============================
from amlh.session import run_preflight_tests
if RUN_TESTS:
    run_preflight_tests(CODE)
    print('all tests passed')

In [ ]:
# ============================= Reuse prior verified work =============================
# The freshly cloned repository's own results/ folder holds every job this project has finished so far, in exactly
# the job-folder layout the runner understands. Pointing it there for free: a job already verified elsewhere is
# never repeated, and a job that was interrupted partway (with a usable checkpoint) resumes instead of restarting.
PRIOR_RESULTS = [CODE / 'results']
print('will reuse any verified job found under', [str(p) for p in PRIOR_RESULTS])

## Extra output to fold into the merge (optional)

If you ran notebooks 01 and/or 02 first and copied or unzipped their output next to this notebook's own working
directory, list those folders here. Leave empty to merge only this session's own NB5 output plus whatever is
already committed in the cloned repository — the notebook still runs correctly either way.

In [ ]:
EXTRA_ROOTS = []   # e.g. [BASE / 'out_nb1', BASE / 'out_nb4'] if you copied them in
EXTRA_ROOTS = [Path(p) for p in EXTRA_ROOTS if Path(p).exists()]
print('extra roots found:', EXTRA_ROOTS or 'none (this is fine; the merge below still runs)')

## Run NB5

Attacks are built on top of a clean-data detector's own fixed thresholds, with a **matched-displacement control**
for every attack condition (`shift_burst`, `shift_stretch`, `shift_blend`) — so a suppression in detection can
be attributed to the *timing pattern itself*, not merely to moving events around at all.

In [ ]:
from amlh.session import run_notebook, print_summary, final_checklist

nb5_work = WORKING / 'nb5'
summary = run_notebook('NB5', tiers=TIERS, budget_hours=hours_left(), input_root=INPUT, working=nb5_work,
                        restore_roots=PRIOR_RESULTS, cache_root=CACHE, run_name='aml_run')
print_summary(nb5_work)
final_checklist('NB5', TIERS, nb5_work, platform='molab')


## Timing-robustness verdicts (C6, C8) and attack plots

C6: does a burst evade a strict-past detector, and does a same-minute view blunt this? C8: does logical-clock
evidence resist `stretch` and `blend` where wall-clock evidence does not, at an acceptable clean-data cost?

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from amlh.merge import collect, timing_verdicts, attack_table
from amlh.jobs_gbt import get_stream

c = collect([nb5_work] + PRIOR_RESULTS + EXTRA_ROOTS)
s, _ = get_stream({'dataset': 'HI-Small', 'protocol': 'released'})
tv = timing_verdicts(c, s)
print('Timing-robustness verdicts (HI-Small, exploratory -- see notebook 02 for why HI-Small is not confirmatory):')
for hid, v in sorted(tv.items()):
    print(f"  {hid}: {'PASS' if v.get('pass') else 'FAIL'}  {json.dumps({k: v[k] for k in v if k != 'pass'}, default=str)[:200]}")
(WORKING / 'timing_verdicts.json').write_text(json.dumps(tv, indent=1, default=str))

wall = attack_table(c, s, 'wall', '0', 21600)
if wall:
    conds = sorted(wall)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(conds, [wall[cd]['detected'].mean() for cd in conds])
    ax.set_ylabel('fraction of attempts detected'); ax.set_title('Wall-clock detector under each attack condition, H=0')
    plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.savefig(WORKING / 'attack_detection.png', dpi=130); plt.show()


## Final merge: unseal, compute every verdict, build the benchmark report

This is the one place in the whole project sealed test metrics are computed, and it happens **at most once per
dataset per code version** — `amlh.sealing.UnsealLog` permanently records every unsealing, and a repeat under
different code is labelled `exploratory`, never silently treated as a fresh confirmation.

In [ ]:
from amlh.merge import merge

ALL_ROOTS = [nb5_work] + PRIOR_RESULTS + EXTRA_ROOTS
DATASETS = ['HI-Small', 'LI-Small']   # add 'SAML-D', 'HI-Medium', 'LI-Medium' here once notebook 02 has swept them

MERGE_OUT = WORKING / 'merged'
report = merge(ALL_ROOTS, MERGE_OUT, DATASETS, analysis_root=CODE)
print((MERGE_OUT / 'MERGE_REPORT.md').read_text())


## Benchmark comparison against the published baseline

`published_reference` below is the official Multi-GNN paper's own reported minority-class F1 on HI-Small (as cited
in this project's literature review, `paper/cn_phase2.tex`), via `amlh.merge.PUBLISHED_F1`. Read this table with
the project's actual goal in mind: **this project is not trying to out-accuracy the baseline model** —
`docs/PHASE3_PLAN.md` is explicit that reproducing accuracy is not among its claims. Matching the published number
under full look-ahead (large H) is an expected sanity check. The actual contribution is the *gap* as H shrinks
toward 0: a genuine, measured answer to "how much of the published number is unrealistic look-ahead", which the
original paper's own evaluation protocol never reports.

In [ ]:
from amlh.merge import official_audit
Paths_report = []
for ds in ('HI-Small',):
    s, _ = get_stream({'dataset': ds, 'protocol': 'released'})
    audit = official_audit(c, s)
    for jid, a in sorted(audit.items()):
        if 'problem' in a or a.get('published_reference') is None:
            continue
        r = a['row']
        gap = a['official_style_f1'] - a['published_reference']
        print(f"{jid:42s} H={str(a['H']):>8s}  reproduced={a['official_style_f1']:.4f}  "
              f"published={a['published_reference']:.4f}  gap={gap:+.4f}")
print()
print('A full-window (H=inf) job close to the published number, alongside an H=0 (strict-past) job well below it,')
print('is this project\'s central empirical finding: direct evidence that the published number embeds look-ahead.')


## Regenerate the paper's tables and figures

`scripts/08_analyze.py` turns the exported results above into exactly the tables and figures `paper/cn_phase2.tex`
includes — nothing in the paper's results is typed by hand.

In [ ]:
import subprocess, shutil, os as _os
link = CODE / 'data' / 'processed'
link.parent.mkdir(parents=True, exist_ok=True)
if not link.exists():
    try:
        _os.symlink(CACHE, link)
    except OSError:
        shutil.copytree(CACHE, link, dirs_exist_ok=True)   # symlinks need a privilege Molab may not grant; copy instead
result = subprocess.run([sys.executable, str(CODE / 'scripts' / '08_analyze.py'), str(MERGE_OUT)], capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print('--- stderr ---'); print(result.stderr[-2000:])
if (CODE / 'results' / 'report').exists():
    shutil.copytree(CODE / 'results' / 'report', MERGE_OUT / 'report', dirs_exist_ok=True)
    print('tables and figures copied to', MERGE_OUT / 'report')


In [ ]:
# ============================= Package the output =============================
# Molab sessions are not guaranteed to persist, so the output is zipped here (and the notebook's own cell outputs,
# saved with the .ipynb itself, carry the printed tables below even if the zip is never downloaded).
import shutil
archive = shutil.make_archive(str(BASE / f'attacks_and_merge_{NOTEBOOK.lower()}'), 'zip', WORKING)
print('wrote', archive, f'({Path(archive).stat().st_size / 1e6:.1f} MB)')
print()
print('Download this file before your Molab session ends. If you plan to run another of the three notebooks next,')
print('the most useful thing to carry over is the whole', WORKING, 'folder (or this zip, unpacked next to the new')
print("notebook's own working directory) -- the next notebook's merge step will pick up anything placed under")
print("its PRIOR_RESULTS / EXTRA_ROOTS list.")

## Completion summary

Read `MERGE_REPORT.md` above (and `merged.json` for the full machine-readable detail) for the authoritative status:
which hypotheses have a verdict, which datasets are unsealed, and the plan-completion counts for every one of the
five original notebooks. Download the zip this notebook wrote — it is this project's most complete single
artifact so far.